# PolygroveDB — generate the REAL card catalog (step 24)

nb22 regenerated the 15 demo species. This scales it: pull EVERY species that
supports all six data-backed levels (Taxonomy + Feeding + Size + Food web +
Habitat/Origin/Biome + Is-this-your-card), straight from the DB — no hand seed.
Efficient/set-based: precompute each signal once, window the food web top-N.


In [ ]:
import duckdb, json, time
con = duckdb.connect("../../polygrove.duckdb")
con.sql(r"""CREATE OR REPLACE MACRO norm(s) AS
  lower(array_to_string(list_filter(string_split(regexp_replace(coalesce(s,''),'\([^)]*\)','','g'),' '),
  x -> x <> '' AND lower(x) NOT IN ('subsp.','ssp.','var.','f.','subvar.'))[1:2],' '));""")
t0=time.time()

# evidence-counted food-web edges (restricted later to the flagship set)
con.sql("""CREATE OR REPLACE TEMP TABLE edges AS
SELECT ns.species_id pred_sid, nt.species_id prey_sid, count(*) evidence,
       count(*) FILTER (WHERE i.interactionTypeName='preysOn') AS predation
FROM interactions i JOIN name2sid ns ON ns.norm_name=norm(i.sourceTaxonName)
JOIN name2sid nt ON nt.norm_name=norm(i.targetTaxonName)
WHERE i.interactionTypeName IN ('eats','preysOn') AND ns.species_id<>nt.species_id GROUP BY 1,2;""")

# per-signal sets
# feeding guild, PRIORITY-ranked so a good source always wins the bucket:
# 1 = measured trait DBs (AVONET/Elton/ReptTraits + plant/fungi floors),
# 2 = TetrapodTraits diet, 3 = GloBI-derived (from prey kingdom) as last resort.
con.sql("""CREATE OR REPLACE TEMP TABLE feed AS
SELECT sid, arg_min(t, pri) AS t, arg_min(src, pri) AS src, arg_min(conf, pri) AS conf FROM (
  SELECT species_id sid, CASE WHEN niche IN ('Herbivore','Herbivore terrestrial','Herbivore aquatic','Granivore','Frugivore','Nectarivore') THEN 'Herbivore'
       WHEN niche='Omnivore' THEN 'Omnivore' WHEN niche IN ('Vertivore','Invertivore','Aquatic predator','Scavenger','Planktivore','Carnivore') THEN 'Carnivore' END AS t, 1 AS pri, 'trait database' AS src, 'measured' AS conf
  FROM vert_niche WHERE niche IS NOT NULL
  UNION ALL SELECT species_id, CASE role WHEN 'producer' THEN 'Producer' WHEN 'decomposer' THEN 'Decomposer' END, 1, 'taxonomy floor', 'definitional'
  FROM species_trophic_roles WHERE role IN ('producer','decomposer')
  UNION ALL SELECT species_id, CASE role WHEN 'carnivore' THEN 'Carnivore' WHEN 'herbivore' THEN 'Herbivore' WHEN 'omnivore' THEN 'Omnivore' END, 1, 'EOL TraitBank', 'measured'
  FROM species_trophic_roles WHERE role IN ('carnivore','herbivore','omnivore')
  UNION ALL SELECT species_id, niche, 2, 'TetrapodTraits', 'measured' FROM tetrapod_traits WHERE niche IS NOT NULL
  UNION ALL SELECT species_id, niche, 3, 'GloBI (from recorded prey)', 'inferred' FROM globi_feeding WHERE niche IS NOT NULL) sub
WHERE t IS NOT NULL GROUP BY sid;""")
# SIZE — only UNIT-TAGGED sources so a same-group comparison is like-with-like
# (each group uses one measure: birds/mammals/reptiles=g, fish/plants=cm, amph=mm).
# Raw EOL species_size is DROPPED: it's unitless and mixes mass with length, which
# makes "which is larger" nonsense (comparing mm to g).
con.sql("""CREATE OR REPLACE TEMP TABLE sz AS SELECT sid, any_value(src) src, any_value(val) val FROM (
  SELECT species_id sid, 'AVONET' src, mass::varchar||' g' val FROM bird_traits WHERE mass>0
  UNION ALL SELECT n.species_id,'EltonTraits', m.mass::varchar||' g' FROM mammal_traits m JOIN name2sid n ON n.norm_name=norm(m.sciname) WHERE m.mass>0
  UNION ALL SELECT species_id,'ReptTraits', mass_g::varchar||' g' FROM reptile_traits WHERE mass_g>0
  UNION ALL SELECT species_id,'AmphiBIO', body_size_mm::varchar||' mm' FROM amphibian_traits WHERE body_size_mm>0
  UNION ALL SELECT species_id,'EOL-height', round(height)::varchar||' cm' FROM plant_fungi_size WHERE height>0
  UNION ALL SELECT species_id,'TetrapodTraits', size_val FROM tetrapod_traits WHERE size_val IS NOT NULL
  UNION ALL SELECT species_id,'AnimalTraits', mass_g::varchar||' g' FROM animaltraits_size WHERE mass_g>0) GROUP BY sid;""")
con.sql("""CREATE OR REPLACE TEMP TABLE hab AS SELECT sid, any_value(src) src, any_value(val) val FROM (
  SELECT species_id sid,'AVONET' src, habitat val FROM bird_traits WHERE habitat IS NOT NULL
  UNION ALL SELECT species_id,'ReptTraits',habitat FROM reptile_traits WHERE habitat IS NOT NULL AND habitat<>'None'
  UNION ALL SELECT species_id,'AmphiBIO',habitat FROM amphibian_traits WHERE habitat IS NOT NULL
  UNION ALL SELECT species_id,'TetrapodTraits',habitat FROM tetrapod_traits WHERE habitat IS NOT NULL) GROUP BY sid;""")
con.sql("CREATE OR REPLACE TEMP TABLE orig AS SELECT species_id sid, string_agg(native_continent,', ') val FROM species_native_origin GROUP BY 1;")
# obligate carnivores — their "prey" that is a PLANT is perching/foraging-substrate
# noise (a flycatcher recorded "eating" the tree it hunts insects on), not diet.
con.sql("""CREATE OR REPLACE TEMP TABLE carn AS
  SELECT species_id FROM vert_niche WHERE niche IN ('Invertivore','Vertivore','Aquatic predator','Carnivore')
  UNION SELECT species_id FROM species_trophic_roles WHERE role='carnivore';""")
# a real "eats" edge = NOT (carnivore eating a non-animal)
con.sql("""CREATE OR REPLACE TEMP TABLE eats_ok AS
  SELECT e.pred_sid, e.prey_sid, e.evidence, e.predation
  FROM edges e JOIN species tp ON tp.species_id=e.prey_sid
  WHERE NOT (e.pred_sid IN (SELECT species_id FROM carn) AND tp.kingdom<>'Animalia');""")
# food-web eligible = has real prey OR a real "eaten by" edge (kingdom-aware).
con.sql("""CREATE OR REPLACE TEMP TABLE fwset AS
  SELECT DISTINCT pred_sid sid FROM eats_ok
  UNION SELECT e.prey_sid FROM edges e JOIN species s ON s.species_id=e.prey_sid
    WHERE (s.kingdom='Animalia' AND e.predation>0) OR (s.kingdom<>'Animalia' AND e.evidence>0);""")
con.sql("CREATE OR REPLACE TEMP TABLE biome AS SELECT DISTINCT species_id sid FROM species_ecoregions;")
# Q6 is now the same-family-nearest-location fallback (CrypticBio dropped — CC-BY-NC),
# so its eligibility is simply "has a location" (biome).

# ALL cards: taxonomy (free) + >=3 of the 5 content levels = a genuinely playable
# card. Each carries whatever levels it has (flagship all-6 is just a subset).
con.sql("""CREATE OR REPLACE TABLE catalog AS
SELECT * FROM (
  SELECT s.species_id, s.canonicalName, s.class, s.kingdom,
    (s.species_id IN (SELECT sid FROM feed))::int
    + (s.species_id IN (SELECT sid FROM sz))::int
    + (s.species_id IN (SELECT sid FROM fwset))::int
    + ((s.species_id IN (SELECT sid FROM hab)) OR (s.species_id IN (SELECT sid FROM orig)) OR (s.species_id IN (SELECT sid FROM biome)))::int
    + (s.species_id IN (SELECT sid FROM biome))::int AS levels
  FROM species s WHERE s.class IS NOT NULL)
WHERE levels = 5;""")   # all 5 content levels present = every card has 6 questions
N=con.sql("SELECT count(*) FROM catalog").fetchone()[0]
print(f"catalog (every card = all 6 questions): {N:,} species  ({time.time()-t0:.0f}s)")
con.sql("SELECT levels+1 AS total_levels, count(*) n FROM catalog GROUP BY 1 ORDER BY 1 DESC").show()


## Precompute lookups + windowed food web (top-4 each side by evidence)


In [ ]:
def as_map(sql):
    return {r[0]: r[1] for r in con.sql(sql).fetchall()}
NAME = as_map("SELECT species_id, canonicalName FROM species")   # display name (GBIF canonical; common-name backfill from GBIF vernacular = TODO)
FEED = as_map("SELECT sid, t FROM feed")
FEED_SRC = {r[0]:(r[1],r[2]) for r in con.sql("SELECT sid,src,conf FROM feed").fetchall()}  # sid -> (source, confidence)
SZ   = {r[0]:(r[1],r[2]) for r in con.sql("SELECT sid,src,val FROM sz").fetchall()}
HAB  = {r[0]:(r[1],r[2]) for r in con.sql("SELECT sid,src,val FROM hab").fetchall()}
ORIG = as_map("SELECT sid, val FROM orig")
BIOME = as_map("""SELECT se.species_id, arg_max(e.ECO_NAME, se.occurrence_count)
  FROM species_ecoregions se JOIN ecoregions e ON e.ECO_ID = se.ecoregion_id
  WHERE se.species_id IN (SELECT species_id FROM catalog) GROUP BY 1""")
cat_ids = [r[0] for r in con.sql("SELECT species_id FROM catalog").fetchall()]
catset = set(cat_ids)
CARDABLE = set(r[0] for r in con.sql("""SELECT DISTINCT species_id FROM (
  SELECT species_id FROM vert_niche WHERE niche IS NOT NULL UNION SELECT species_id FROM species_trophic_roles)""").fetchall())

# windowed top-4 eats + top-4 eaten-by for the whole catalog at once
con.sql("""CREATE OR REPLACE TEMP TABLE fw_eats AS
  SELECT pred_sid sid, prey_sid partner, evidence FROM eats_ok WHERE pred_sid IN (SELECT species_id FROM catalog)
  QUALIFY row_number() OVER (PARTITION BY pred_sid ORDER BY evidence DESC)<=4;""")
con.sql("""CREATE OR REPLACE TEMP TABLE fw_eb AS
  SELECT prey_sid sid, pred_sid partner, ev AS evidence FROM (
    SELECT e.prey_sid, e.pred_sid,
      CASE WHEN s.kingdom='Animalia' THEN e.predation ELSE e.evidence END AS ev
    FROM edges e JOIN species s ON s.species_id=e.prey_sid
    WHERE e.prey_sid IN (SELECT species_id FROM catalog))
  WHERE ev>0
  QUALIFY row_number() OVER (PARTITION BY prey_sid ORDER BY ev DESC)<=4;""")
from collections import defaultdict
EATS=defaultdict(list); EB=defaultdict(list)
for sid,p,e in con.sql("SELECT sid,partner,evidence FROM fw_eats").fetchall(): EATS[sid].append((p,e))
for sid,p,e in con.sql("SELECT sid,partner,evidence FROM fw_eb").fetchall():   EB[sid].append((p,e))
# Q6 look-alikes: 2 same-FAMILY species sharing the card's ecoregion (closest
# local relatives ≈ what people actually confuse). Pure GBIF taxonomy + RESOLVE
# ecoregions — both CC-BY. Replaces the dropped CC-BY-NC CrypticBio confusions.
LA=defaultdict(list)
for sid,p in con.sql("""
WITH cat_fe AS (
  SELECT c.species_id, s.family,
    (SELECT ecoregion_id FROM species_ecoregions se WHERE se.species_id=c.species_id
     ORDER BY occurrence_count DESC LIMIT 1) AS eco
  FROM catalog c JOIN species s ON s.species_id=c.species_id WHERE s.family IS NOT NULL
), cand AS (
  SELECT f.species_id, o.species_id AS partner,
    row_number() OVER (PARTITION BY f.species_id ORDER BY o.species_id) rn
  FROM cat_fe f
  JOIN species_ecoregions se ON se.ecoregion_id=f.eco
  JOIN species o ON o.species_id=se.species_id AND o.family=f.family AND o.species_id<>f.species_id
  WHERE f.eco IS NOT NULL
)
SELECT species_id, partner FROM cand WHERE rn<=2""").fetchall():
    LA[sid].append(p)


## Emit the catalog JSON


In [ ]:
GROUP={'Gastropoda':'Mollusk','Bivalvia':'Mollusk','Cephalopoda':'Mollusk','Mammalia':'Mammal','Aves':'Bird',
 'Amphibia':'Amphibian','Squamata':'Reptile','Testudines':'Reptile','Crocodylia':'Reptile','Actinopterygii':'Fish',
 'Chondrichthyes':'Fish','Insecta':'Insect','Arachnida':'Arachnid','Agaricomycetes':'Fungus','Magnoliopsida':'Plant',
 'Liliopsida':'Plant','Pinopsida':'Plant','Polypodiopsida':'Plant'}
import random
_GW  = {'Carnivore':'a carnivore','Herbivore':'a herbivore','Omnivore':'an omnivore'}
_OTH = {'Carnivore':['an omnivore','a herbivore'],'Herbivore':['an omnivore','a carnivore'],
        'Omnivore':['a carnivore','a herbivore']}
def q2_framing(sid, trophic, isapex):
    # Q2 feeding-guild true/false. Producer<->decomposer are each other's trap.
    # Apex is orthogonal to guild: randomly "an apex predator" OR its REAL guild
    # (omnivore for a bear, carnivore for a condor); trap = a guild it isn't.
    rng = random.Random(sid*7+1)
    if trophic=='Producer':   return ("a producer","a decomposer")
    if trophic=='Decomposer': return ("a decomposer","a producer")
    guild = _GW.get(trophic, f"a {str(trophic).lower()}" if trophic else "a carnivore")
    false_word = rng.choice(_OTH.get(trophic,['an omnivore']))
    if isapex and rng.random()<0.5:
        return ("an apex predator", false_word)
    return (guild, false_word)
def q4_dir(sid, trophic, has_eats, has_eb):
    # Plants can't eat -> eaten-by; apex isn't eaten -> eats; fungus/middle -> random.
    if trophic=='Producer' or not has_eats: return 'eatenBy'
    if not has_eb: return 'eats'
    return 'eats' if random.Random(sid*13+3).random()<0.5 else 'eatenBy'
def slugify(name): return name.lower().replace(' ','-').replace("'","")
def node(sid,e): return {"name":NAME.get(sid,str(sid)),"evidence":e,"card":sid in CARDABLE,"inSet":sid in catset}

cards=[]
for sid, canon, cls, kingdom, levels in con.sql("SELECT species_id,canonicalName,class,kingdom,levels FROM catalog").fetchall():
    trophic=FEED.get(sid)
    nm = NAME.get(sid) or canon or str(sid)
    eats=[node(p,e) for p,e in EATS.get(sid,[])]
    eb  =[node(p,e) for p,e in EB.get(sid,[])]
    isapex = trophic!='Producer' and not eb                      # nothing eats it
    role, falseRole = q2_framing(sid, trophic, isapex)
    d = q4_dir(sid, trophic, bool(eats), bool(eb))
    q4pool = eats if d=='eats' else eb
    slot5type = 'habitat' if sid in HAB else ('origin' if sid in ORIG else 'biome')
    feed_src, feed_conf = FEED_SRC.get(sid, (None, None))
    # provenance per field: source + confidence tier (measured | definitional |
    # inferred | derived) so the card's "source" popup is honest about what's
    # known vs guessed — the scientific-database principle.
    prov = {
      "taxonomy": {"source":"GBIF Backbone", "confidence":"measured"},
      "feeding":  {"source": feed_src, "confidence": feed_conf},
      "size":     {"source": SZ.get(sid,(None,))[0], "confidence":"measured"},
      "slot5":    {"source": (HAB.get(sid,(None,))[0] if slot5type=='habitat' else 'WCVP' if slot5type=='origin' else 'GBIF/GARD/PHYLACINE occurrences → RESOLVE ecoregions'),
                   "confidence": "measured" if slot5type in ('habitat','origin') else "derived"},
      "foodweb":  {"source":"GloBI", "confidence":"measured"},
      "lookAlikes": {"source":"GBIF taxonomy + RESOLVE ecoregions", "confidence":"inferred"},
    }
    cards.append({
      "id": slugify(nm), "species_id": sid, "name": nm, "sci": canon,
      "levels": levels+1,
      "group": GROUP.get(cls,cls), "trophic": trophic, "isApex": isapex,
      "role": role, "falseRole": falseRole,                      # Q2 feeding-guild T/F
      "size": SZ.get(sid,(None,None))[1], "habitat": HAB.get(sid,(None,None))[1],
      "origin": ORIG.get(sid), "biome": BIOME.get(sid),
      "slot5": HAB.get(sid,(None,None))[1] or ORIG.get(sid) or BIOME.get(sid), "slot5type": slot5type,
      "foodweb": {"eats":eats, "eatenBy":eb},
      "q4": {"direction": d,                                     # Q4 food-web (fixed at generation)
             "verb": "most known to eat" if d=='eats' else "most known to be eaten by",
             "answer": q4pool[0]['name'] if q4pool else None},
      "lookAlikes": [NAME.get(p,str(p)) for p in LA.get(sid,[])],
      "provenance": prov,
    })
json.dump(cards, open("catalog.json","w"), ensure_ascii=False)
print(f"wrote catalog.json — {len(cards):,} cards ({time.time()-t0:.0f}s total)")
con.sql("SELECT s.kingdom, s.class, count(*) n FROM catalog s GROUP BY 1,2 ORDER BY 3 DESC LIMIT 12").show()
print("look-alikes present on:", sum(1 for c in cards if c['lookAlikes']), "cards")
print("\n=== 3 sample cards ===")
for c in cards[:3]:
    print(f"\n{c['name']} ({c['sci']}) — {c['group']}/{c['trophic']}, size {c['size']}, habitat {c['habitat']}, origin {c['origin']}")
    print(f"  eats: {[n['name'] for n in c['foodweb']['eats']]}")
    print(f"  eaten by: {[n['name'] for n in c['foodweb']['eatenBy']]}   look-alikes: {c['lookAlikes']}")
con.close()
